# 분기(Multi) Chain
- LangChain + Chroma + LLM
- 문서 분할 → 임베딩 → CharomDB저장 → 질문 기반 문서 검색 → Prompt 완성 → LLM 답변 생성
- 요즘은 MCP까지 나옴.

In [ ]:
!pip install langchain langchain-openai langchain-community langchain-core python-dotenv
!pip install langchain-chroma sentence-transformers langchain-text-splitters
!pip install -U langsmith # https://smith.langchain.com/

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda  # 함수를 체인으로 묶기 위한 래퍼
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import  CharacterTextSplitter
import langsmith

# Document : 단순한 텍스트 묶음이 아니라 '정보 단위를 표현하기 위한 구조체'이다.
# 각각의 Document는 검색 가능한 하나의 문서 단위(chunk)임
# 여러 Document로 나누는 이유는 1) 검색 성능 향상. 2)서로 다른 출처/메타데이터가 있음

import os
from dotenv import load_dotenv

'''
[RunnableLambda 사용 예시]
def func(text):
  return text.upper()
obj = RunnableLambda(func) # 함수를 감쌈
print(obj.invoke("hello"))
'''

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY가 없어요")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature="0.5")

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# embedding_model = OpenAIEmbeddings(model="all-MiniLM-L6-v2")

/tmp/ipykernel_43931/1504644574.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_43931/1504644574.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# 실험용 문서 Text파일 : LangChain은 모든 문서 타입을 일반적으로 Document() 형태로 Wrapping해서 작업함.
docs = [
    Document(
        page_content="""
        랭체인(LangChain)은 해리슨 체이스(Harrison Chase)에 의해 2022년 10월에 오픈 소스 프로젝트로 시작되었습니다.
        그는 머신러닝 스타트업인 로버스트 인텔리전스(Robust Intelligence)에서 근무하면서 대규모 언어 모델(LLM)을 활용하여 애플리케이션과 파이프라인을 신속하게 구축할 수 있는 플랫폼의 필요성을 느꼈습니다.
        이러한 비전을 가지고 개발자들이 챗봇, 질의응답 시스템, 자동 요약 등 다양한 LLM 애플리케이션을 쉽게 개발할 수 있도록 지원하는 프레임워크를 만들었습니다.
        1. AI 추론 및 생성 계층
        다양한 LLM 모델(OpenAI, Anthropic, Google 등)을 표준화된 인터페이스로 사용할 수 있습니다. 특정 공급자에 종속되지 않고 자유롭게 모델을 교체할 수 있습니다.
        2. 외부 기능 계층
        API, 데이터베이스, 웹 서비스 등 외부 시스템과 연동할 수 있습니다. Tool Calling을 통해 LLM이 정의된 함수를 직접 호출할 수 있습니다.
        3. 오케스트레이션 계층
        LangGraph를 기반으로 하는 에이전트(Agent)를 통해 복잡한 추론과 행동의 흐름을 제어합니다. 내구성 있는 실행(durable execution), 스트리밍, 영속성을 지원합니다.
        4. 컨텍스트 보존 계층
        메시지 히스토리, 커스텀 상태 스키마, 장기 메모리를 통해 대화의 맥락을 유지합니다. 토큰 한계를 넘지 않으면서 효과적으로 컨텍스트를 관리할 수 있습니다.
        5. 정보 접근 계층
        문서 로더, 텍스트 분할기, 벡터 스토어, 검색기(Retriever)를 통해 RAG(Retrieval-Augmented Generation) 패턴을 구현할 수 있습니다.
        """,
    ),
    Document(
        page_content="""
        RAG는 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합한 자연어 처리(NLP) 기술을 의미합니다.
        전통적인 생성 모델과는 달리, RAG는 먼저 데이터베이스나 문서 집합에서 관련 정보를 검색하고, 검색한 정보를 바탕으로 텍스트를 생성합니다.
        """
    ),
]

# Text를 작은 조각으로 분리 - RAG에서 권장
text_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separator="\n\n", # 기본값: 행을 기준으로 자름
)
# RecursiveCharacterTextSplitter는 구분자를 재귀적으로 여러개를 사용

split_docs = text_splitter.split_documents(docs)
print("분할된 문서 수 :", len(split_docs))
print("분할된 문서 첫번째 :", split_docs[0].page_content)

# 쪼개진 문서를 Vectord data로 변환해서(embedding) ChromaDB에 넣기
db = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    # persist_directory="./mydb",
    # collection_name="mycollection",
)

# Retriver : 사용자의 질문과 유사한 문서를 Chroma에서 검색하는 객체 - langChain은 추상화가 되어 있다.
retriver = db.as_retriever()

분할된 문서 수 : 2
분할된 문서 첫번째 : 랭체인(LangChain)은 해리슨 체이스(Harrison Chase)에 의해 2022년 10월에 오픈 소스 프로젝트로 시작되었습니다. 
        그는 머신러닝 스타트업인 로버스트 인텔리전스(Robust Intelligence)에서 근무하면서 대규모 언어 모델(LLM)을 활용하여 애플리케이션과 파이프라인을 신속하게 구축할 수 있는 플랫폼의 필요성을 느꼈습니다.
        이러한 비전을 가지고 개발자들이 챗봇, 질의응답 시스템, 자동 요약 등 다양한 LLM 애플리케이션을 쉽게 개발할 수 있도록 지원하는 프레임워크를 만들었습니다.
        1. AI 추론 및 생성 계층
        다양한 LLM 모델(OpenAI, Anthropic, Google 등)을 표준화된 인터페이스로 사용할 수 있습니다. 특정 공급자에 종속되지 않고 자유롭게 모델을 교체할 수 있습니다.
        2. 외부 기능 계층
        API, 데이터베이스, 웹 서비스 등 외부 시스템과 연동할 수 있습니다. Tool Calling을 통해 LLM이 정의된 함수를 직접 호출할 수 있습니다.
        3. 오케스트레이션 계층
        LangGraph를 기반으로 하는 에이전트(Agent)를 통해 복잡한 추론과 행동의 흐름을 제어합니다. 내구성 있는 실행(durable execution), 스트리밍, 영속성을 지원합니다.
        4. 컨텍스트 보존 계층
        메시지 히스토리, 커스텀 상태 스키마, 장기 메모리를 통해 대화의 맥락을 유지합니다. 토큰 한계를 넘지 않으면서 효과적으로 컨텍스트를 관리할 수 있습니다.
        5. 정보 접근 계층
        문서 로더, 텍스트 분할기, 벡터 스토어, 검색기(Retriever)를 통해 RAG(Retrieval-Augmented Generation) 패턴을 구현할 수 있습니다.


In [ ]:
# Prompt Template 구성
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template = """
      너는 친절하고 똑똑한 AI 어시스턴트야.
      아래 문서 내용을 참고해서 사용자의 질문에 답해 줘.
      문서 내용이 불 충분한 경우에는 '문서에 해당 정보가 없어요'라고 답변해.

      문서 내용:
      {context}

      질문:
      {question}

      답변은 5행 정도의 분량이면 좋겠어.
    """
)

In [ ]:
# Chain 구성 : LCEL 사용
def format_docs(docs):
    # doc.page_content : Document의 속성을 불러옴(추상화 되어 있어서 속성들을 알고 있어야 함)
    return "\n\n".join(doc.page_content for doc in docs)
rag_chain = (
    {
        # 질문 유사 문장 얻기 -> retriever로 검색 -> 문서들을 문자열로 포맷
        "context": retriver | RunnableLambda(format_docs),

        # 질문이 들어오면 전혀 변형 없이 질문을 그대로 변수로 넘김(Prompt의 question으로 매핑됨)
        "question": RunnablePassthrough(),

    } # prompt_template을 여기에서 완성
    | prompt_template   # LangChain Pipline ↓
    | llm
    | StrOutputParser()
)

# 질문 생성
query = 'RAG가 뭔가요?'

# 질문들 들고 rag_chain을 호출
result = rag_chain.invoke(query)
print("질문 :",query)
print("답변 :",result)

질문 : RAG가 뭔가요?
답변 : RAG는 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합한 자연어 처리 기술입니다. 기존 생성 모델과 달리, RAG는 먼저 데이터베이스나 문서 집합에서 관련 정보를 검색합니다. 그런 다음, 검색한 정보를 바탕으로 텍스트를 생성하는 방식입니다. 이를 통해 더 정확하고 풍부한 답변을 만들 수 있습니다. LangChain에서는 RAG 패턴을 구현하는 데 필요한 문서 로더, 텍스트 분할기, 벡터 스토어, 검색기 등을 제공합니다.


In [ ]:
# 조건에 따른 분기 처리 - Tool없이 해보기
# RunnableBranch 사용
from langchain_core.runnables import RunnableBranch

# 조건 관련 함수
def is_weathre_question(text:str) -> bool:
  return "날씨" in text.lower()

# 분기 A : 날씨 관련 질문
weather_chain = RunnableLambda(lambda x:"오늘의 날씨는 흐리고 최고 온도는 32도입니다")

# 분기 B : 일반 질문
basic_general_chain = RunnableLambda(lambda x:f"일반 질문이군요.'{x}'에 대해 설명할게요")

# Branch 조합 - 분기 처리 체인
basic_branch_chain = RunnableBranch(
    (is_weathre_question, weather_chain), # 조건이 참일때
    basic_general_chain,                  # 조건이 거짓일때
)

# 실행 테스트
print(basic_branch_chain.invoke("오늘 날씨는 어때?"))
print(basic_branch_chain.invoke("점심 메뉴 추천해줘"))

오늘의 날씨는 흐리고 최고 온도는 32도입니다
일반 질문이군요.'점심 메뉴 추천해줘'에 대해 설명할게요


In [ ]:
# 조건에 따른 LLM 수행 - 원래는 GPT, Gemini, Claude등의 LLM모델을 조건에 따라 선택하게 함.
llm2 = ChatOpenAI(model="gpt-4o-mini", temperature="0.0")

# Prompt 생성 함수
# 수학 질문용
def make_math_prompt(question:str) -> str:
  return(
      "당신은 수학풀이를 잘하는 전문가입니다.\n"\
      "아래 수학 문제를 단계별로 푸는 과정을 적어주고, 마지막 줄에 정답만 한번도 적어 줘.\n\n"\
      f"문제 : {question}\n\n"\
      "풀이:"
  )

  # 코딩 질문용
def make_code_prompt(question:str) -> str:
  return(
      "당신은 프로그래머 강사다.\n"\
      "아래 요청에 대해 1)간단한 설명, 2)예제 코드, 3) 중요 포인트 순서대로 말해줘.\n\n"\
      f"요청 : {question}\n\n"\
      "답변:"
  )

  # 일반 질문용
def make_general_prompt(question:str) -> str:
  return(
      "당신은 친절한 AI 설명가입니다.\n"\
      "아래 질문에 대해 초보자도 이해할 수 있도록 5문장 정도로 답을 줘.\n\n"\
      f"질문 : {question}\n\n"\
      "설명:"
  )

# question을 채워야 하므로 LCL방식 사용
# 체인 구성 : 질문 -> Prompt생성 -> LLM호출 -> 결과 출력
math_chain = (RunnableLambda(make_math_prompt)
| llm2
| StrOutputParser())

code_chain = (RunnableLambda(make_code_prompt)
| llm2
| StrOutputParser())

general_chain = (RunnableLambda(make_general_prompt)
| llm
| StrOutputParser())

# 분기조건 함수들 생성
# 질문에 수학 관련 유무 확인
def is_math_question(text:str) -> bool:
  t = text.replace(" ","").lower()
  math_keywords = [
      "더하기","빼기","곱하기","나누기","계산","합","차","곱","나눈값","몇","얼마"
  ]
  math_symbols = ["+","-","*","/","^"]
  return any(k in t for k in math_keywords) or any(s in t for s in math_symbols)

# 질문에 코딩 관련 유무 확인
def is_code_question(text:str) -> bool:
  t = text.lower()
  code_keywords = [
      "코드","함수","클래스","메소드","알고리즘","파이썬","python","자바","java"
  ]
  return any(k in t for k in code_keywords)

# 분기처리 Chain생성
branch_chain = RunnableBranch(
    (is_math_question, math_chain),
    (is_code_question, code_chain),
    general_chain,
)

In [ ]:
# 실행
questions = [
    '3 + 5 * (2)^2 는 얼마야',
    '파이썬으로 리스트 정렬하는 코드 만들어줘',
    '짬뽕이야 짜장이야?'
]

for q in questions:
  print("\n질문 :",q)
  result = branch_chain.invoke(q)
  print("답변 :",result)


질문 : 3 + 5 * (2)^2 는 얼마야
답변 : 문제를 단계별로 풀어보겠습니다.

1. **괄호 안의 계산**: 
   - (2)^2 = 2 * 2 = 4

2. **곱셈 계산**: 
   - 5 * 4 = 20

3. **덧셈 계산**: 
   - 3 + 20 = 23

따라서, 3 + 5 * (2)^2 의 값은 23입니다. 

정답: 23

질문 : 파이썬으로 리스트 정렬하는 코드 만들어줘
답변 : 1) **간단한 설명**: 
파이썬에서 리스트를 정렬하는 방법은 여러 가지가 있지만, 가장 일반적으로 사용되는 방법은 `sort()` 메서드와 `sorted()` 함수를 사용하는 것입니다. `sort()` 메서드는 리스트 자체를 정렬하며, `sorted()` 함수는 정렬된 새로운 리스트를 반환합니다. 두 방법 모두 기본적으로 오름차순으로 정렬하지만, 내림차순으로 정렬할 수도 있습니다.

2) **예제 코드**:

```python
# 리스트 생성
numbers = [5, 2, 9, 1, 5, 6]

# 1. sort() 메서드 사용 (리스트 자체를 정렬)
numbers.sort()
print("오름차순 정렬 (sort()):", numbers)

# 2. sorted() 함수 사용 (새로운 리스트 반환)
numbers = [5, 2, 9, 1, 5, 6]  # 원래 리스트로 복원
sorted_numbers = sorted(numbers)
print("오름차순 정렬 (sorted()):", sorted_numbers)

# 3. 내림차순 정렬
numbers.sort(reverse=True)
print("내림차순 정렬 (sort()):", numbers)

# 4. sorted()로 내림차순 정렬
numbers = [5, 2, 9, 1, 5, 6]  # 원래 리스트로 복원
sorted_numbers_desc = sorted(numbers, reverse=True)
print("내림차순 정렬 (sorted()):", sorted_numbers_desc)
```

In [ ]:
# LangSmith